# Synchronise Eye-Tracking & Body-Tracking Data

Loads one participant's processed CSVs from both modalities and interpolates
BT onto the ET timeline within each `(participant_id, condition_number, trial_number)` group.

## Strategy

| | Eye-tracking | Body-tracking |
|---|---|---|
| File | `001_cleaned_ET.csv` | `001_cleaned_BT.csv` |
| Folder | `data/eye_tracking/processed/` | `data/body_tracking/processed/` |
| Approx. rate | ~200 Hz | ~90 Hz |
| Alignment clock | `raw_timestamp` (ms) | `raw_timestamp` (ms) |

**Continuous BT columns** (positions, rotations, velocities) → linear interpolation onto ET timestamps.  
**Discrete BT columns** (`model_name`, `bad_sample_*`, `is_interpolated_*`, etc.) → nearest-neighbour passthrough.  
All BT columns are prefixed `bt_` in the output.


# 1. Imports & Configuration

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

ET_PROCESSED_DIR = Path("../data/eye_tracking/processed")
BT_PROCESSED_DIR = Path("../data/body_tracking/processed")
OUTPUT_DIR       = Path("../data/merged")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GROUP_COLS = ["participant_id", "condition_number", "trial_number"]


# 2. Helper Functions

## 2.1 Load

In [2]:
def normalize_pid(participant_id):
    pid = str(participant_id).strip()
    if not pid.isdigit():
        raise ValueError(f"Participant ID must be numeric, got: {participant_id!r}")
    return pid.zfill(3)


def load_cleaned_pair(participant_id,
                      et_dir=ET_PROCESSED_DIR,
                      bt_dir=BT_PROCESSED_DIR):
    pid = normalize_pid(participant_id)
    et_path = Path(et_dir) / f"{pid}_cleaned_ET.csv"
    bt_path = Path(bt_dir) / f"{pid}_cleaned_BT.csv"

    missing = [str(p) for p in (et_path, bt_path) if not p.exists()]
    if missing:
        raise FileNotFoundError("Missing file(s):\n- " + "\n- ".join(missing))

    et_df = pd.read_csv(et_path, low_memory=False)
    bt_df = pd.read_csv(bt_path, low_memory=False)

    for frame in (et_df, bt_df):
        if "participant_id" in frame.columns:
            frame["participant_id"] = frame["participant_id"].astype(str).str.zfill(3)
        if "raw_timestamp" not in frame.columns:
            raise KeyError("Both files must contain 'raw_timestamp'.")
        frame["raw_timestamp"] = pd.to_numeric(frame["raw_timestamp"], errors="coerce")

    if "gaze_capture_time" not in et_df.columns:
        raise KeyError("ET file must contain 'gaze_capture_time'.")
    et_df["gaze_capture_time"] = pd.to_numeric(et_df["gaze_capture_time"], errors="coerce")

    print(f"ET  {pid}: {len(et_df):>7,} rows  |  {et_path.name}")
    print(f"BT  {pid}: {len(bt_df):>7,} rows  |  {bt_path.name}")

    return pid, et_path, bt_path, et_df, bt_df


## 2.2 Sampling Rate Summaries

In [3]:
def et_sampling_rate_summary(df):
    t     = pd.to_numeric(df["gaze_capture_time"], errors="coerce").dropna().sort_values()
    dt_ns = t.diff().dropna()
    dt_ns = dt_ns[dt_ns > 0]
    median_dt_ns = dt_ns.median()
    return pd.DataFrame([{
        "stream":                  "ET",
        "time_col":                "gaze_capture_time",
        "n_rows":                  len(t),
        "unique_timestamps":       t.nunique(),
        "median_dt":               median_dt_ns,
        "median_dt_unit":          "ns",
        "median_sampling_rate_hz": 1e9 / median_dt_ns if pd.notna(median_dt_ns) else np.nan,
    }])


def bt_sampling_rate_summary(df):
    t     = pd.to_numeric(df["raw_timestamp"], errors="coerce").dropna().sort_values()
    dt_ms = t.diff().dropna()
    dt_ms = dt_ms[dt_ms > 0]
    median_dt_ms = dt_ms.median()
    return pd.DataFrame([{
        "stream":                  "BT",
        "time_col":                "raw_timestamp",
        "n_rows":                  len(t),
        "unique_timestamps":       t.nunique(),
        "median_dt":               median_dt_ms,
        "median_dt_unit":          "ms",
        "median_sampling_rate_hz": 1000.0 / median_dt_ms if pd.notna(median_dt_ms) else np.nan,
    }])


## 2.3 BT Column Classification

In [13]:
def get_bt_column_groups(bt_df, et_df=None, group_cols=GROUP_COLS):
    exclude = {"raw_timestamp", *group_cols}

    if et_df is not None:
        duplicate_cols = set(bt_df.columns).intersection(et_df.columns)
        exclude.update(duplicate_cols)

    discrete_passthrough = {
        "model_name", "model_rot_deg",
        "LeftFootArea", "RightFootArea",
        "source_file", "segment_label",
        "sampling_rate_hz", "dt", "dt_ms",
        "participant_id",
    }

    continuous_cols = []
    discrete_cols = []

    for col in bt_df.columns:
        if col in exclude:
            continue

        if col.startswith("bad_sample_") or col.startswith("is_interpolated_"):
            discrete_cols.append(col)
        elif col in discrete_passthrough:
            discrete_cols.append(col)
        elif pd.api.types.is_numeric_dtype(bt_df[col]):
            continuous_cols.append(col)
        else:
            discrete_cols.append(col)

    print(f"  Continuous BT cols ({len(continuous_cols)}): {continuous_cols[:5]}{'...' if len(continuous_cols) > 5 else ''}")
    print(f"  Discrete   BT cols ({len(discrete_cols)}):   {discrete_cols[:5]}{'...' if len(discrete_cols) > 5 else ''}")

    return continuous_cols, discrete_cols

## 2.4 Interpolation

In [14]:
def prepare_bt_segment(bt_segment, continuous_cols, discrete_cols):
    bt_seg = bt_segment.copy()
    bt_seg["raw_timestamp"] = pd.to_numeric(bt_seg["raw_timestamp"], errors="coerce")
    bt_seg = bt_seg.dropna(subset=["raw_timestamp"]).sort_values("raw_timestamp")

    agg_map = {}
    for col in continuous_cols:
        if col in bt_seg.columns:
            agg_map[col] = "mean"
    for col in discrete_cols:
        if col in bt_seg.columns:
            agg_map[col] = "first"

    if not agg_map:
        raise ValueError("No BT columns available for synchronisation.")

    return (
        bt_seg.groupby("raw_timestamp", as_index=False, dropna=False)
        .agg(agg_map)
        .sort_values("raw_timestamp")
        .reset_index(drop=True)
    )


def interpolate_bt_to_et(et_segment, bt_segment, continuous_cols, discrete_cols):
    et_sync = et_segment.copy().sort_values("raw_timestamp").reset_index(drop=True)
    et_t    = pd.to_numeric(et_sync["raw_timestamp"], errors="coerce").to_numpy(dtype=float)

    bt_unique = prepare_bt_segment(bt_segment, continuous_cols, discrete_cols)
    src_t     = bt_unique["raw_timestamp"].to_numpy(dtype=float)

    # nearest-neighbour index for discrete columns
    nearest_idx     = None
    nearest_support = np.zeros(len(et_sync), dtype=bool)

    if len(src_t) >= 1:
        right_idx   = np.searchsorted(src_t, et_t, side="left")
        prev_idx    = np.clip(right_idx - 1, 0, len(src_t) - 1)
        next_idx    = np.clip(right_idx,     0, len(src_t) - 1)
        prev_dist   = np.abs(et_t - src_t[prev_idx])
        next_dist   = np.abs(et_t - src_t[next_idx])
        nearest_idx = np.where(prev_dist <= next_dist, prev_idx, next_idx)
        nearest_support = (
            np.isfinite(et_t) &
            (et_t >= src_t[0]) &
            (et_t <= src_t[-1])
        )

    # continuous: linear interpolation
    for col in continuous_cols:
        out_col = f"bt_{col}"
        if col not in bt_unique.columns:
            et_sync[out_col] = np.nan
            continue
        src_v = pd.to_numeric(bt_unique[col], errors="coerce").to_numpy(dtype=float)
        valid = np.isfinite(src_t) & np.isfinite(src_v)
        if valid.sum() < 2:
            et_sync[out_col] = np.nan
            continue
        t_v, v_v    = src_t[valid], src_v[valid]
        within      = np.isfinite(et_t) & (et_t >= t_v[0]) & (et_t <= t_v[-1])
        interp_vals = np.full(len(et_sync), np.nan, dtype=float)
        interp_vals[within] = np.interp(et_t[within], t_v, v_v)
        et_sync[out_col] = interp_vals

    # discrete: nearest-neighbour passthrough
    for col in discrete_cols:
        out_col = f"bt_{col}"
        if col not in bt_unique.columns:
            et_sync[out_col] = np.nan
            continue
        src_series = bt_unique[col]
        if pd.api.types.is_numeric_dtype(src_series):
            carried = np.full(len(et_sync), np.nan, dtype=float)
            if nearest_idx is not None and nearest_support.any():
                src_values = pd.to_numeric(src_series, errors="coerce").to_numpy(dtype=float)
                carried[nearest_support] = src_values[nearest_idx[nearest_support]]
        else:
            carried = np.full(len(et_sync), np.nan, dtype=object)
            if nearest_idx is not None and nearest_support.any():
                src_values = src_series.astype(object).to_numpy()
                carried[nearest_support] = src_values[nearest_idx[nearest_support]]
        et_sync[out_col] = carried

    return et_sync


def synchronize_et_bt(et_df, bt_df, group_cols=GROUP_COLS):
    missing = [c for c in group_cols if c not in et_df.columns or c not in bt_df.columns]
    if missing:
        raise KeyError(f"Missing grouping column(s): {missing}")

    continuous_cols, discrete_cols = get_bt_column_groups(bt_df, group_cols=group_cols)

    bt_groups = {
        key if isinstance(key, tuple) else (key,): sub.copy()
        for key, sub in bt_df.groupby(group_cols, sort=True, dropna=False)
    }

    empty_bt       = bt_df.iloc[0:0].copy()
    synced_segments = []

    for key, et_segment in et_df.groupby(group_cols, sort=True, dropna=False):
        key = key if isinstance(key, tuple) else (key,)
        bt_segment = bt_groups.get(key, empty_bt)
        synced_segments.append(
            interpolate_bt_to_et(et_segment, bt_segment, continuous_cols, discrete_cols)
        )

    return pd.concat(synced_segments, axis=0, ignore_index=True)


## 2.5 Full Build

In [15]:
def build_synchronized_dataframe(participant_id,
                                  et_dir=ET_PROCESSED_DIR,
                                  bt_dir=BT_PROCESSED_DIR):
    pid, et_path, bt_path, et_df, bt_df = load_cleaned_pair(
        participant_id, et_dir=et_dir, bt_dir=bt_dir
    )

    et_pre = et_sampling_rate_summary(et_df)
    bt_pre = bt_sampling_rate_summary(bt_df)

    sync_df  = synchronize_et_bt(et_df, bt_df)
    et_post  = et_sampling_rate_summary(sync_df)

    et_pre_hz  = et_pre["median_sampling_rate_hz"].iloc[0]
    et_post_hz = et_post["median_sampling_rate_hz"].iloc[0]

    rate_report = pd.DataFrame([
        {"stage": "pre_sync",  "stream": "ET", "median_sampling_rate_hz": et_pre_hz},
        {"stage": "pre_sync",  "stream": "BT", "median_sampling_rate_hz": bt_pre["median_sampling_rate_hz"].iloc[0]},
        {"stage": "post_sync", "stream": "ET", "median_sampling_rate_hz": et_post_hz},
    ])
    rate_report["rate_diff_from_et_pre_hz"] = (
        rate_report["median_sampling_rate_hz"] - et_pre_hz
    )

    meta = {
        "pid":        pid,
        "et_path":    et_path,
        "bt_path":    bt_path,
        "et_shape":   et_df.shape,
        "bt_shape":   bt_df.shape,
        "sync_shape": sync_df.shape,
        "et_pre_hz":  et_pre_hz,
        "et_post_hz": et_post_hz,
    }

    return sync_df, rate_report, meta


# 3. Run for Participant 001

In [20]:
PARTICIPANT_ID = "001"

sync_df, rate_report, meta = build_synchronized_dataframe(PARTICIPANT_ID)

print("\n── Shapes ──────────────────────────────")
print(f"  ET input  : {meta['et_shape']}")
print(f"  BT input  : {meta['bt_shape']}")
print(f"  Merged    : {meta['sync_shape']}")
print("\n── Sampling rates ──────────────────────")
print(rate_report.to_string(index=False))

sync_df.head(3)


ET  001: 646,964 rows  |  001_cleaned_ET.csv
BT  001: 164,974 rows  |  001_cleaned_BT.csv
  Continuous BT cols (70): ['relative_to_unix_epoch_timestamp', 'RightFoot_pos_x', 'RightFoot_pos_y', 'RightFoot_pos_z', 'RightFoot_rot_x']...
  Discrete   BT cols (18):   ['LeftHand_grabbed_name', 'RightHand_grabbed_name', 'model_name', 'model_rot_deg', 'LeftFootArea']...

── Shapes ──────────────────────────────
  ET input  : (646964, 72)
  BT input  : (164974, 92)
  Merged    : (646964, 160)

── Sampling rates ──────────────────────
    stage stream  median_sampling_rate_hz  rate_diff_from_et_pre_hz
 pre_sync     ET               199.972004                  0.000000
 pre_sync     BT                90.909091               -109.062913
post_sync     ET               199.972004                  0.000000


,gaze_capture_time,raw_timestamp,relative_to_unix_epoch_timestamp,focus_distance,frame_number,stability,status,gaze_forward_x,gaze_forward_y,gaze_forward_z,...,bt_is_interpolated_RightFoot,bt_bad_sample_LeftFoot,bt_is_interpolated_LeftFoot,bt_bad_sample_Waist,bt_is_interpolated_Waist,bt_bad_sample_LeftHand,bt_is_interpolated_LeftHand,bt_bad_sample_RightHand,bt_is_interpolated_RightHand,bt_segment_label
0,1000000972356065800,1778059755031,83.28207,0.782032,130142,1.0,Valid,0.024471,0.236849,0.971238,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1000000972361063200,1778059755040,83.28207,0.780995,130143,1.0,Valid,0.024434,0.236810,0.971249,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1000000972366064600,1778059755040,83.28207,0.779816,130144,1.0,Valid,0.024407,0.236764,0.971261,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 4. Save

In [8]:
out_path = OUTPUT_DIR / f"{meta['pid']}_synced.csv"
sync_df.to_csv(out_path, index=False)
print(f"✅  Saved → {out_path}  ({len(sync_df):,} rows, {sync_df.shape[1]} columns)")


✅  Saved → ..\data\merged\001_synced.csv  (646,964 rows, 160 columns)
